# `all_variants.ipynb` — unified pipeline for `colab_qwen25_05b`

**Model:** `Qwen/Qwen2.5-0.5B`   **Samples / class:** `300`

Single notebook that produces, for one model, the **full ablation matrix** over 12 feature variants. Resilient to Colab session timeouts: every expensive stage writes its output to disk, and every stage checks the file exists before running. If the kernel restarts mid-way, just `Run All` again — completed stages are skipped automatically.

## Stages

| # | Stage | Output (skipped if present) | Cost |
|---|---|---|---|
| 1 | Setup + load model | (in-memory only) | ~10–30 s |
| 2 | **MIND data generation** | `colab_qwen25_05b_dataset_full.json` | minutes |
| 3 | **F1–F10 feature extraction** | `colab_qwen25_05b_dataset_with_features.json` | 1–5 min |
| 4 | Train 12 MLP variants | `colab_qwen25_05b_variant_<X>_best.pth` × 12 | <2 min total |
| 5 | Consolidated results dump | `colab_qwen25_05b_all_variants_results.json` | <1 s |

## Feature catalogue

Per sample, after stage 3 the dataset record holds:

* `embedding` — canonical MIND last-token last-layer hidden state (D-dim)
* `D_mean` — mean cosine distance between adjacent layers (drift)
* `V_last` — L2 cross-layer variance of last-token activations
* `H_mean` — mean Shannon entropy of per-step token distribution
* `F1_lookback_ratio` — attention mass on prompt vs generated tokens (Lookback Lens, EMNLP 2024)
* `F2_attention_sink` — fraction of attention mass on BOS/punctuation across layers
* `F3_eigenscore_lite` — log-det of last-layer hidden-state covariance (INSIDE, ICLR 2024)
* `F4_icr_score` — per-layer ratio of module-update norm to residual stream norm (ICR Probe, ACL 2025)
* `F5_logit_lens_jsd` — JSD between intermediate-layer and final-layer next-token distributions (DoLa / SLED)
* `F6_head_entropy` — Shannon entropy of each head's attention at last token, mean over heads
* `F7_max_margin` — mean (top-1 − top-2) softmax-prob gap (HaMI, NeurIPS 2025)
* `F8_token_rank` — mean rank of each emitted token under the greedy distribution
* `F10_intra_dispersion` — mean L2 distance from last-layer centroid across tokens (D²HScore)

(F9 from the menu — SAPLMA-style mid-layer probe — needs a separate supervised training step on a labelled dataset, so it's deferred.)

## Variants trained (edit `VARIANTS` in Stage 4 to add more)

| Variant | Features added on top of canonical |
|---|---|
| A | (none — canonical only) |
| B | D_mean |
| C | V_last |
| D | H_mean |
| E | D_mean + V_last + H_mean (MIND+ headline) |
| F | F1 |
| G | F5 |
| H | F7 |
| I | F1 + F5 + F7 (recommended trio) |
| J | F1 + F2 + F3 + F4 + F5 + F6 + F7 + F8 + F10 (all 9 F-features) |
| K | D_mean + V_last + H_mean + F1 + F5 + F7 (E + trio) |
| L | everything (all 12 scalars) |

## Output JSON shape (`colab_qwen25_05b_all_variants_results.json`)

```jsonc
{
  "schema_version": "all-variants-1.0",
  "model_tag": "colab_qwen25_05b",
  "model_name": "Qwen/Qwen2.5-0.5B",
  "config": {...},
  "env": {...},
  "stage_timings_sec": {...},
  "data_gen": {... skip counters etc ...},
  "feature_stats": { "F1": {mean,std,min,max}, ... "F10": {...} },
  "variants": {
    "A": {"input_dim": ..., "feature_keys": [...],
          "wikipedia_eval": {accuracy, precision, recall, f1, auc_roc, brier,
                             cm, label_distribution_test, pred_distribution_test, ...},
          "mlp_epoch_history": [{epoch, train_loss, test_acc}, ...]},
    "B": {...}, ... "L": {...}
  },
  "errors": [...]
}
```

Paste this single JSON back to the assistant after the run for cross-variant audit.


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS  (Colab already has most; pin minima.)
# =============================================================================
!pip install -q -U "transformers>=4.45" "tokenizers>=0.19" "accelerate>=0.30"
!pip install -q -U datasets spacy nltk scikit-learn tqdm sentence-transformers
!python -m spacy download en_core_web_sm


In [ ]:
# =============================================================================
# BLOCK 1: SETUP — imports, config, diagnostics
# =============================================================================
import os, sys, gc, json, random, math, time, datetime, platform, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MODEL_NAME       = "Qwen/Qwen2.5-0.5B"
MODEL_TAG        = "colab_qwen25_05b"
TARGET_SAMPLES   = 300       # per class
TOPK_FIRST_TOKEN = 4
WINDOWS          = 16
SEED             = 42
DTYPE            = torch.bfloat16 if torch.cuda.is_available() else torch.float32
LOGIT_LENS_LAYER = 12       # for F5 (DoLa-style early-exit reference)
MAX_GEN_NEW      = 48
EPOCHS           = 10
BATCH            = 32
LR               = 5e-4
WD               = 1e-5
TEST_FRAC        = 0.20

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# File paths (every stage's output is one of these — resume points)
PATH_DATASET_FULL     = f"{MODEL_TAG}_dataset_full.json"
PATH_DATASET_FEATURES = f"{MODEL_TAG}_dataset_with_features.json"
PATH_RESULTS          = f"{MODEL_TAG}_all_variants_results.json"
PATH_CHECKPOINT       = f"{MODEL_TAG}_datagen_checkpoint.json"

# Diagnostics dict — grown across the whole notebook and dumped at end
DIAG = {
    "schema_version": "all-variants-1.0",
    "model_tag":  MODEL_TAG,
    "model_name": MODEL_NAME,
    "config": {
        "target_samples_per_class": TARGET_SAMPLES,
        "topk_first_token": TOPK_FIRST_TOKEN, "windows": WINDOWS, "seed": SEED,
        "dtype": str(DTYPE), "logit_lens_layer": LOGIT_LENS_LAYER,
        "max_gen_new_tokens": MAX_GEN_NEW,
        "epochs": EPOCHS, "batch_size": BATCH,
        "lr": LR, "weight_decay": WD, "test_frac": TEST_FRAC,
    },
    "env": {
        "python":   sys.version.split()[0],
        "platform": platform.platform(),
        "torch":    torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device":   (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        "free_vram_gb":  (round(torch.cuda.mem_get_info()[0]/1e9, 2) if torch.cuda.is_available() else None),
    },
    "library_versions": {},
    "stage_timings_sec": {},
    "errors": [],
}
for _mod in ["transformers", "datasets", "accelerate", "sklearn", "spacy", "nltk", "numpy"]:
    try:
        m = __import__(_mod)
        DIAG["library_versions"][_mod] = getattr(m, "__version__", "n/a")
    except Exception as e:
        DIAG["library_versions"][_mod] = f"IMPORT_FAILED: {e}"

# We only load the LLM on demand (saves time if data files already exist)
_MODEL = {"tokenizer": None, "model": None, "nlp": None}

def need_llm():
    """Lazy-load the LLM + spacy. Idempotent."""
    if _MODEL["model"] is not None:
        return _MODEL
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import spacy, nltk
    nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
    print(f"Loading {MODEL_NAME} (dtype={DTYPE}) ...")
    _t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    load_kwargs = dict(torch_dtype=DTYPE,
                       device_map="auto" if torch.cuda.is_available() else None)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    nlp = spacy.load("en_core_web_sm")
    DIAG["stage_timings_sec"]["model_load"] = round(time.perf_counter() - _t0, 2)
    DIAG["model_info"] = {
        "hidden_dim":    int(mdl.config.hidden_size),
        "n_layers":      int(mdl.config.num_hidden_layers),
        "n_heads":       int(getattr(mdl.config, "num_attention_heads", -1)),
        "vocab_size":    int(mdl.config.vocab_size),
        "max_pos_embed": int(getattr(mdl.config, "max_position_embeddings", -1)),
        "param_count_M": round(sum(p.numel() for p in mdl.parameters()) / 1e6, 2),
        "device":        str(next(mdl.parameters()).device),
    }
    print(f"✓ loaded in {DIAG['stage_timings_sec']['model_load']}s   "
          f"hidden_dim={DIAG['model_info']['hidden_dim']}   "
          f"layers={DIAG['model_info']['n_layers']}")
    _MODEL["tokenizer"] = tok; _MODEL["model"] = mdl; _MODEL["nlp"] = nlp
    return _MODEL


print("=" * 80); print(f"BLOCK 1: SETUP — {MODEL_TAG}"); print("=" * 80)
print(f"Output files (skipped if already present):")
print(f"  Stage 2 ->  {PATH_DATASET_FULL}      {'EXISTS' if os.path.exists(PATH_DATASET_FULL) else 'will be generated'}")
print(f"  Stage 3 ->  {PATH_DATASET_FEATURES}  {'EXISTS' if os.path.exists(PATH_DATASET_FEATURES) else 'will be generated'}")
print(f"  Stage 5 ->  {PATH_RESULTS}           {'EXISTS' if os.path.exists(PATH_RESULTS) else 'will be generated'}")


In [ ]:
# =============================================================================
# BLOCK 2 (STAGE 2): MIND DATA GENERATION  (skipped if dataset_full.json exists)
# Algorithm 1 from Su et al. (Findings of ACL 2024). Saves checkpoint every
# 200 samples so a session disconnect mid-gen does not waste GPU time.
# =============================================================================
if os.path.exists(PATH_DATASET_FULL):
    print(f"[SKIP STAGE 2] Loading existing {PATH_DATASET_FULL} ...")
    _t0 = time.perf_counter()
    with open(PATH_DATASET_FULL, "r") as f:
        records_full = json.load(f)
    DIAG["stage_timings_sec"]["data_gen_skipped_load"] = round(time.perf_counter() - _t0, 2)
    n_h1 = sum(1 for r in records_full if r["label"] == 1)
    n_h0 = sum(1 for r in records_full if r["label"] == 0)
    DIAG["data_gen"] = {"loaded_from_disk": True,
                         "n_hallucinated": n_h1, "n_non_hallucinated": n_h0,
                         "n_total": len(records_full)}
    print(f"  {len(records_full)} records  (y=1:{n_h1}  y=0:{n_h0})")
else:
    print(f"Generating MIND dataset to {PATH_DATASET_FULL} ...")
    LL = need_llm()
    tokenizer, model, nlp = LL["tokenizer"], LL["model"], LL["nlp"]

    from datasets import load_dataset
    from nltk.tokenize import sent_tokenize
    from tqdm.auto import tqdm

    # -- entity helpers --
    def delete_substrings(lst):
        subs = []; lst = list(set(lst))
        for s in lst:
            if any(s in o for o in lst if o != s): subs.append(s)
        for s in subs: lst.remove(s)
        return lst

    def find_boundaries(text, words):
        out = []
        for word in words:
            ntext = text
            while True:
                start = ntext.find(word)
                if start == -1: break
                end = start + len(word) - 1
                while start > 0 and ntext[start-1] != " ": start -= 1
                while end < len(ntext)-1 and ntext[end+1] != " ": end += 1
                out.append("".join(ntext[i] for i in range(start, end+1)))
                ntext = ntext[end+1:]
        return out

    def get_entities(text):
        ents = list({str(e) for e in nlp(text).ents})
        ents = find_boundaries(text, ents)
        ents = delete_substrings(ents)
        out = []
        for i in range(len(text)):
            for e in ents:
                if text[i:].startswith(e): out.append((e, i))
        return out

    def find_first_and_next_token(text, e, idx, input_id, prompt=""):
        new_text = f"{text[:idx].strip()} {text[idx:].replace(e, e+' @', 1).strip()}"
        new_input_id = tokenizer(prompt + new_text.strip(), return_tensors="pt")["input_ids"].tolist()[0]
        for i in range(len(input_id[0])):
            if input_id[0][i] != new_input_id[i]: return []
        first_token = new_input_id[len(input_id[0])]
        at_cands = tokenizer("@", add_special_tokens=False)["input_ids"]
        at_cands += tokenizer(" @", add_special_tokens=False)["input_ids"]
        at_pos = None
        for at_tok in at_cands:
            try:
                at_pos = new_input_id.index(at_tok, len(input_id[0])); break
            except ValueError: continue
        if at_pos is None or at_pos >= len(new_input_id) - 1: return []
        next_token = new_input_id[at_pos + 1]
        entity_len = at_pos - len(input_id[0])
        last_id = new_input_id[at_pos + 1:]
        return [first_token, next_token, entity_len, last_id]

    @torch.no_grad()
    def extract_mind(text: str):
        """canonical (last token last layer), D_mean, V_last, H_last."""
        enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                        max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        hs = out.hidden_states
        last_per_layer = torch.stack([h[0, -1, :].float() for h in hs[1:]], dim=0)
        canonical = last_per_layer[-1].cpu().tolist()
        if last_per_layer.shape[0] < 2:
            D_mean = 0.0
        else:
            cos = F.cosine_similarity(last_per_layer[:-1], last_per_layer[1:], dim=1)
            D_mean = float((1.0 - cos).mean().item())
        mean_h = last_per_layer.mean(dim=0)
        V_last = float(((last_per_layer - mean_h) ** 2).sum(dim=1).mean().item())
        last_logits = out.logits[0, -1, :].float()
        p_last = torch.softmax(last_logits, dim=-1)
        H_last = float(-(p_last * p_last.clamp_min(1e-12).log()).sum().item())
        return canonical, D_mean, V_last, H_last

    def per_step_entropy(scores):
        if not scores: return 0.0
        Hs = []
        for s in scores:
            p = torch.softmax(s[0].float(), dim=-1)
            Hs.append(float(-(p * p.clamp_min(1e-12).log()).sum().item()))
        return float(sum(Hs) / len(Hs))

    SKIP = {"overflow_prefix": 0, "no_token_match": 0, "overflow_suffix": 0,
             "model_knew": 0, "no_match_in_topk": 0, "empty_entity": 0,
             "duplicate_entity": 0, "exception": 0}

    def generate_sample(text, entity, idx, title):
        MAX_POS = getattr(model.config, "max_position_embeddings", 4096)
        HEADROOM = WINDOWS + 64
        enc = tokenizer(text[:idx].strip(), return_tensors="pt").to(model.device)
        input_ids = enc["input_ids"]; attn = enc["attention_mask"]
        if input_ids.shape[1] + HEADROOM >= MAX_POS:
            SKIP["overflow_prefix"] += 1; return None
        input_id_list = input_ids.tolist()
        toks = find_first_and_next_token(text, entity, idx, input_id_list)
        if not toks: SKIP["no_token_match"] += 1; return None
        first_, next_, entity_len, last_id = toks
        if input_ids.shape[1] + entity_len + len(last_id) + HEADROOM >= MAX_POS:
            SKIP["overflow_suffix"] += 1; return None
        gen = model.generate(input_ids, attention_mask=attn,
                             max_new_tokens=entity_len + WINDOWS,
                             return_dict_in_generate=True, output_scores=True,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
        if first_ in torch.topk(gen.scores[0], k=TOPK_FIRST_TOKEN).indices[0].tolist():
            SKIP["model_knew"] += 1; return None
        found_step = None
        for step, sc in enumerate(gen.scores):
            if next_ in torch.topk(sc, k=TOPK_FIRST_TOKEN).indices[0].tolist():
                found_step = step; break
        if found_step is None:
            SKIP["no_match_in_topk"] += 1; return None
        H_mean_hall = per_step_entropy(gen.scores[: found_step + 1])
        new_seq = gen.sequences[0, : input_ids.shape[1] + found_step].tolist()
        new_entity_ids = new_seq[input_ids.shape[1]:]
        hallucinated_ids = input_id_list[0] + new_entity_ids + last_id
        hallucinated_text = tokenizer.decode(hallucinated_ids, skip_special_tokens=True)
        new_entity = tokenizer.decode(new_entity_ids, skip_special_tokens=True).strip().lower()
        if not new_entity: SKIP["empty_entity"] += 1; return None
        if entity.lower() in new_entity or new_entity in text.lower():
            SKIP["duplicate_entity"] += 1; return None
        canon_h, D_h, V_h, _    = extract_mind(hallucinated_text)
        canon_o, D_o, V_o, H_o  = extract_mind(text)
        return {"text_hall": hallucinated_text, "entity_hall": new_entity,
                "canon_h": canon_h, "D_h": D_h, "V_h": V_h, "H_h": H_mean_hall,
                "text_orig": text, "entity_orig": entity,
                "canon_o": canon_o, "D_o": D_o, "V_o": V_o, "H_o": H_o,
                "title": title}

    print("Loading Wikipedia (streaming) ...")
    wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    dataset_hall, dataset_non_hall = [], []
    processed = 0
    pbar = tqdm(total=TARGET_SAMPLES * 2, desc="samples")
    _t0 = time.perf_counter()
    for row in wiki:
        if len(dataset_hall) >= TARGET_SAMPLES and len(dataset_non_hall) >= TARGET_SAMPLES:
            break
        processed += 1
        try:
            sents = sent_tokenize(row["text"])
            if len(sents) < 2: continue
            text = " ".join(sents[:2])
            title = row.get("title", "")
            ents = get_entities(text)
            if not ents: continue
            seen, ents_filt = set(), []
            for e, i in ents:
                if i == 0 or e.lower() in title.lower(): continue
                if i not in seen:
                    seen.add(i); ents_filt.append((e, i))
            if not ents_filt: continue
            entity, char_idx = random.choice(ents_filt)
            result = generate_sample(text, entity, char_idx, title)
            if result is None: continue
            if len(dataset_hall) < TARGET_SAMPLES:
                dataset_hall.append({"label": 1, "text": result["text_hall"],
                                      "entity": result["entity_hall"],
                                      "embedding": result["canon_h"],
                                      "D_mean": result["D_h"], "V_last": result["V_h"],
                                      "H_mean": result["H_h"], "title": result["title"]})
                pbar.update(1)
            if len(dataset_non_hall) < TARGET_SAMPLES:
                dataset_non_hall.append({"label": 0, "text": result["text_orig"],
                                          "entity": result["entity_orig"],
                                          "embedding": result["canon_o"],
                                          "D_mean": result["D_o"], "V_last": result["V_o"],
                                          "H_mean": result["H_o"], "title": result["title"]})
                pbar.update(1)
            pbar.set_postfix(H1=len(dataset_hall), H0=len(dataset_non_hall), proc=processed)
            if (len(dataset_hall) + len(dataset_non_hall)) % 200 == 0:
                with open(PATH_CHECKPOINT, "w") as f:
                    json.dump(dataset_hall + dataset_non_hall, f)
        except Exception as e:
            SKIP["exception"] += 1; continue
        if processed % 100 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    pbar.close()
    DIAG["stage_timings_sec"]["data_gen"] = round(time.perf_counter() - _t0, 2)

    records_full = dataset_hall + dataset_non_hall
    with open(PATH_DATASET_FULL, "w") as f:
        json.dump(records_full, f)

    DIAG["data_gen"] = {
        "loaded_from_disk": False,
        "wiki_articles_processed": processed,
        "n_hallucinated": len(dataset_hall),
        "n_non_hallucinated": len(dataset_non_hall),
        "n_total": len(records_full),
        "skip_counters": dict(SKIP),
        "example_hall":     (dataset_hall[0]["text"][:280]     if dataset_hall else None),
        "example_non_hall": (dataset_non_hall[0]["text"][:280] if dataset_non_hall else None),
    }
    print(f"\n✓ wrote {PATH_DATASET_FULL}  ({os.path.getsize(PATH_DATASET_FULL)/1e6:.2f} MB)")
    print(f"  data_gen time: {DIAG['stage_timings_sec']['data_gen']} s")
    print(f"  skip counters: {SKIP}")


In [ ]:
# =============================================================================
# BLOCK 3 (STAGE 3): COMPUTE F1..F10 EXTRA FEATURES per record
# (Skipped if dataset_with_features.json already exists.)
# =============================================================================
EXTRA_FEATURE_KEYS = [
    "F1_lookback_ratio",
    "F2_attention_sink",
    "F3_eigenscore_lite",
    "F4_icr_score",
    "F5_logit_lens_jsd",
    "F6_head_entropy",
    "F7_max_margin",
    "F8_token_rank",
    "F10_intra_dispersion",
]

if os.path.exists(PATH_DATASET_FEATURES):
    print(f"[SKIP STAGE 3] Loading existing {PATH_DATASET_FEATURES} ...")
    _t0 = time.perf_counter()
    with open(PATH_DATASET_FEATURES, "r") as f:
        records_full = json.load(f)
    DIAG["stage_timings_sec"]["features_skipped_load"] = round(time.perf_counter() - _t0, 2)
    print(f"  {len(records_full)} records, hidden_dim {len(records_full[0]['embedding'])}")
else:
    print(f"Computing F1..F10 features for {len(records_full)} records ...")
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    N_LAYERS = model.config.num_hidden_layers

    def jsd(p, q, eps=1e-12):
        p = p / p.sum(); q = q / q.sum()
        m = 0.5 * (p + q)
        def _kl(a, b):
            return float((a * (a.clamp_min(eps).log() - b.clamp_min(eps).log())).sum().item())
        return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

    # Pre-compute the token-IDs we treat as "attention sinks" for F2
    _SINK_STRS = ["<|endoftext|>", "<s>", "</s>", "<|im_start|>", "<|im_end|>",
                  ".", ",", "!", "?", ";", ":", " .", " ,", " !", " ?", " ;", " :"]
    _SINK_IDS = set()
    for s in _SINK_STRS:
        try:
            ids = tokenizer(s, add_special_tokens=False)["input_ids"]
            for tid in ids: _SINK_IDS.add(int(tid))
        except Exception: continue
    # also include BOS/EOS/PAD if defined
    for tid in [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id]:
        if tid is not None: _SINK_IDS.add(int(tid))

    @torch.no_grad()
    def extract_extras(text):
        """Returns dict with all 9 extra features. Single forward pass."""
        # boundary between "prompt" and "continuation" (heuristic: first ". ")
        idx = text.find(". ")
        prompt_text = text[: (idx + 2 if idx != -1 else len(text)//2)]
        # tokenise full
        enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                        max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
        input_ids = enc.input_ids
        T = input_ids.shape[1]
        p_enc = tokenizer(prompt_text.strip(), return_tensors="pt", truncation=True,
                           max_length=getattr(model.config, "max_position_embeddings", 4096)).input_ids
        prompt_len = max(1, min(p_enc.shape[1], T - 1))
        out = model(**enc, output_hidden_states=True, output_attentions=True, use_cache=False)
        hs = out.hidden_states                    # tuple (L+1)
        attn = out.attentions                     # tuple L of [1, n_heads, T, T]
        logits = out.logits                       # [1, T, V]

        # ----- F1 Lookback Ratio (last layer, last position, mean over heads)
        last_layer_attn = attn[-1][0]             # [n_heads, T, T]
        last_pos = last_layer_attn[:, -1, :]      # [n_heads, T]
        mass_p = last_pos[:, :prompt_len].sum(dim=1).float()
        mass_g = last_pos[:, prompt_len:].sum(dim=1).float()
        F1 = float((mass_p / (mass_p + mass_g).clamp_min(1e-8)).mean().item())

        # ----- F2 Attention-Sink Score (avg across layers + heads, last position)
        sink_mask = torch.zeros(T, dtype=torch.bool, device=input_ids.device)
        ids_list = input_ids[0].tolist()
        for i, tid in enumerate(ids_list):
            if int(tid) in _SINK_IDS: sink_mask[i] = True
        # if no sinks in this sample, F2 defaults to 0
        if sink_mask.any():
            sink_masses = []
            for layer_attn in attn:
                a = layer_attn[0, :, -1, :].float()         # [n_heads, T]
                m = a[:, sink_mask].sum(dim=1) / a.sum(dim=1).clamp_min(1e-8)
                sink_masses.append(float(m.mean().item()))
            F2 = float(sum(sink_masses) / len(sink_masses))
        else:
            F2 = 0.0

        # ----- F3 EigenScore-Lite (log-det of last-layer hidden-state covariance)
        H_last = hs[-1][0].float()                            # [T, D]
        Hc = H_last - H_last.mean(dim=0, keepdim=True)
        # ridge for numerical stability
        ridge = 1e-3
        if T >= 2:
            cov_t = (Hc @ Hc.t()) / max(T - 1, 1)             # [T, T] — smaller dim
            cov_t = cov_t + ridge * torch.eye(T, device=cov_t.device, dtype=cov_t.dtype)
            try:
                sign, logabsdet = torch.slogdet(cov_t)
                F3 = float(logabsdet.item() if sign > 0 else float("nan"))
            except Exception:
                F3 = float("nan")
        else:
            F3 = float("nan")

        # ----- F4 ICR Score (mean across layers of ||h_l − h_{l−1}|| / ||h_l||, last token)
        ratios = []
        for l in range(1, len(hs)):
            h_prev = hs[l-1][0, -1, :].float()
            h_curr = hs[l  ][0, -1, :].float()
            denom = float(h_curr.norm().item())
            if denom <= 1e-8: continue
            ratios.append(float((h_curr - h_prev).norm().item()) / denom)
        F4 = float(sum(ratios) / max(len(ratios), 1))

        # ----- F5 Logit-Lens JSD between layer LOGIT_LENS_LAYER and final layer
        try:
            lm_w = model.get_output_embeddings().weight.float()
            ref_layer = min(LOGIT_LENS_LAYER, len(hs) - 1)
            early_h = hs[ref_layer][0, -1, :].float()
            early_logits = early_h @ lm_w.t()
            p_early = torch.softmax(early_logits, dim=-1)
            p_final = torch.softmax(logits[0, -1, :].float(), dim=-1)
            F5 = jsd(p_early, p_final)
        except Exception:
            F5 = float("nan")

        # ----- F6 Attention-Head Entropy at last token, mean over heads + layers
        head_ents = []
        for layer_attn in attn:
            a = layer_attn[0, :, -1, :].float().clamp_min(1e-12)   # [n_heads, T]
            a = a / a.sum(dim=1, keepdim=True).clamp_min(1e-12)
            H_per_head = -(a * a.log()).sum(dim=1)                 # [n_heads]
            head_ents.append(float(H_per_head.mean().item()))
        F6 = float(sum(head_ents) / max(len(head_ents), 1))

        # ----- F7 Token Max-Margin (mean across positions of top1 − top2 prob)
        if T >= 2:
            probs = torch.softmax(logits[0, :-1, :].float(), dim=-1)    # [T-1, V]
            top2  = probs.topk(2, dim=-1).values
            F7 = float((top2[:, 0] - top2[:, 1]).mean().item())
        else:
            F7 = 0.0

        # ----- F8 Token Rank (mean rank of emitted token t+1 under distribution at t)
        if T >= 2:
            probs = torch.softmax(logits[0, :-1, :].float(), dim=-1)
            tgt = input_ids[0, 1:]                                       # [T-1]
            ranks = []
            # batch-friendly: use argsort and look up positions
            sort_idx = probs.argsort(dim=-1, descending=True)            # [T-1, V]
            for i in range(T - 1):
                pos = (sort_idx[i] == tgt[i]).nonzero(as_tuple=True)
                if len(pos[0]) > 0:
                    ranks.append(float(pos[0][0].item() + 1))
            F8 = float(sum(ranks) / max(len(ranks), 1))
        else:
            F8 = 0.0

        # ----- F10 Intra-Layer Dispersion (mean L2 dist from last-layer centroid)
        if T >= 2:
            H_last_f = hs[-1][0].float()
            centroid = H_last_f.mean(dim=0, keepdim=True)
            F10 = float((H_last_f - centroid).pow(2).sum(dim=1).sqrt().mean().item())
        else:
            F10 = 0.0

        return {
            "F1_lookback_ratio": F1,  "F2_attention_sink":   F2,
            "F3_eigenscore_lite": F3, "F4_icr_score":        F4,
            "F5_logit_lens_jsd":  F5, "F6_head_entropy":     F6,
            "F7_max_margin":      F7, "F8_token_rank":       F8,
            "F10_intra_dispersion": F10,
        }

    _t0 = time.perf_counter()
    failures = 0
    accum = {k: [] for k in EXTRA_FEATURE_KEYS}
    for i, r in enumerate(records_full):
        try:
            extras = extract_extras(r["text"])
        except Exception as e:
            failures += 1
            DIAG["errors"].append(f"extract_extras row {i}: {type(e).__name__}: {e}")
            extras = {k: 0.0 for k in EXTRA_FEATURE_KEYS}
        for k, v in extras.items():
            r[k] = v
            accum[k].append(v)
        if (i + 1) % max(1, len(records_full) // 10) == 0:
            print(f"  {i+1}/{len(records_full)}  F1={extras['F1_lookback_ratio']:.4f}  "
                  f"F5={extras['F5_logit_lens_jsd']:.4f}  F7={extras['F7_max_margin']:.4f}")
    DIAG["stage_timings_sec"]["feature_extraction"] = round(time.perf_counter() - _t0, 2)
    DIAG["n_feature_failures"] = failures
    DIAG["feature_stats"] = {k: {
        "mean": float(np.nanmean(v)) if v else float("nan"),
        "std":  float(np.nanstd(v))  if v else float("nan"),
        "min":  float(np.nanmin(v))  if v else float("nan"),
        "max":  float(np.nanmax(v))  if v else float("nan"),
    } for k, v in accum.items()}
    with open(PATH_DATASET_FEATURES, "w") as f:
        json.dump(records_full, f)
    print(f"\n✓ wrote {PATH_DATASET_FEATURES}  ({os.path.getsize(PATH_DATASET_FEATURES)/1e6:.2f} MB)")
    print(f"  feature_extraction time: {DIAG['stage_timings_sec']['feature_extraction']} s   failures: {failures}")
    for k, st in DIAG["feature_stats"].items():
        print(f"   {k:25s} mean={st['mean']:.4f}  std={st['std']:.4f}")

# Free GPU memory if no further LLM work needed (all variants are CPU-friendly)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
# =============================================================================
# BLOCK 4 (STAGE 4): TRAIN 12 MLP VARIANTS
# Each variant is a different subset of scalar features stacked on canonical.
# Edit `VARIANTS` to add more combinations — the loop will pick them up.
# =============================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, confusion_matrix, brier_score_loss)

ALL_SCALAR_KEYS = ["D_mean", "V_last", "H_mean"] + EXTRA_FEATURE_KEYS

VARIANTS = {
    "A": [],
    "B": ["D_mean"],
    "C": ["V_last"],
    "D": ["H_mean"],
    "E": ["D_mean", "V_last", "H_mean"],
    "F": ["F1_lookback_ratio"],
    "G": ["F5_logit_lens_jsd"],
    "H": ["F7_max_margin"],
    "I": ["F1_lookback_ratio", "F5_logit_lens_jsd", "F7_max_margin"],
    "J": ["F1_lookback_ratio", "F2_attention_sink", "F3_eigenscore_lite",
          "F4_icr_score", "F5_logit_lens_jsd", "F6_head_entropy",
          "F7_max_margin", "F8_token_rank", "F10_intra_dispersion"],
    "K": ["D_mean", "V_last", "H_mean",
          "F1_lookback_ratio", "F5_logit_lens_jsd", "F7_max_margin"],
    "L": ALL_SCALAR_KEYS,
}

# 80/20 split with seed=42 — IDENTICAL across all variants
random.seed(SEED)
records_shuf = list(records_full)
random.shuffle(records_shuf)
split_idx = int((1.0 - TEST_FRAC) * len(records_shuf))
train_recs = records_shuf[:split_idx]
test_recs  = records_shuf[split_idx:]
print(f"Split: train {len(train_recs)}   test {len(test_recs)}")

hidden_dim = len(records_full[0]["embedding"])


class MINDPlusClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.layers(x)


class _FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def build_X_y(recs, keys, scaler=None, fit=False):
    canon = np.array([r["embedding"] for r in recs], dtype=np.float32)
    if keys:
        scalars = np.array([[r.get(k, 0.0) for k in keys] for r in recs], dtype=np.float32)
        # replace NaN with col mean (only an issue for F3 EigenScore on tiny sequences)
        col_means = np.nanmean(scalars, axis=0)
        inds = np.where(np.isnan(scalars))
        scalars[inds] = np.take(col_means, inds[1])
        if fit:
            scaler = StandardScaler().fit(scalars)
        scalars_z = scaler.transform(scalars).astype(np.float32)
        X = np.concatenate([canon, scalars_z], axis=1)
    else:
        X = canon
    y = np.array([r["label"] for r in recs], dtype=np.int64)
    return X, y, scaler


def _safe_auc(y, pr):
    try:    return float(roc_auc_score(y, pr)) if len(set(y.tolist())) > 1 else float("nan")
    except Exception: return float("nan")


def train_one_variant(variant_letter, keys):
    """Train one MLP variant. Returns metrics dict for the variant."""
    # Re-seed so each variant starts identically
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

    X_tr, y_tr, scaler = build_X_y(train_recs, keys, fit=True)
    X_te, y_te, _      = build_X_y(test_recs,  keys, scaler=scaler, fit=False)
    input_dim = X_tr.shape[1]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mlp = MINDPlusClassifier(input_dim).to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(mlp.parameters(), lr=LR, weight_decay=WD)
    train_loader = DataLoader(_FeatureDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
    test_loader  = DataLoader(_FeatureDataset(X_te, y_te), batch_size=BATCH, shuffle=False)

    best_acc = 0.0; epoch_hist = []
    best_state = None
    for ep in range(EPOCHS):
        mlp.train()
        ep_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); lg = mlp(x); l = loss_fn(lg, y); l.backward(); opt.step()
            ep_loss += float(l.item())
        mlp.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                correct += (mlp(x).argmax(1) == y).sum().item(); total += y.size(0)
        acc_ep = correct / max(total, 1)
        epoch_hist.append({"epoch": ep + 1,
                            "train_loss": round(ep_loss / max(len(train_loader), 1), 4),
                            "test_acc": round(acc_ep, 4)})
        if acc_ep > best_acc:
            best_acc = acc_ep
            best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}

    if best_state is not None:
        ckpt = {"model_state": best_state, "input_dim": input_dim,
                "variant": variant_letter, "scalar_keys": keys}
        if keys:
            ckpt["scaler_mean"] = scaler.mean_; ckpt["scaler_scale"] = scaler.scale_
        torch.save(ckpt, f"{MODEL_TAG}_variant_{variant_letter}_best.pth")
        mlp.load_state_dict(best_state)

    mlp.eval()
    all_p, all_y, all_pr = [], [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device); lg = mlp(x)
            prob = torch.softmax(lg, dim=1)[:, 1]
            all_p.extend(lg.argmax(1).cpu().numpy()); all_y.extend(y.numpy())
            all_pr.extend(prob.cpu().numpy())
    all_p = np.array(all_p); all_y = np.array(all_y); all_pr = np.array(all_pr)

    acc  = float(accuracy_score(all_y, all_p)) if len(all_y) else float("nan")
    prec, rec, f1, _ = precision_recall_fscore_support(all_y, all_p, average="binary",
                                                         zero_division=0)
    auc   = _safe_auc(all_y, all_pr)
    brier = float(brier_score_loss(all_y, all_pr)) if len(all_y) else float("nan")
    cm    = confusion_matrix(all_y, all_p, labels=[0,1])

    return {
        "feature_keys": keys,
        "input_dim":    int(input_dim),
        "mlp_best_acc": float(best_acc),
        "mlp_epoch_history": epoch_hist,
        "wikipedia_eval": {
            "n": int(len(all_y)),
            "accuracy":  float(acc), "precision": float(prec),
            "recall":    float(rec), "f1":        float(f1),
            "auc_roc":   float(auc), "brier":     float(brier),
            "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]),
                   "fn": int(cm[1,0]), "tp": int(cm[1,1])},
            "label_distribution_test": {"y=0": int((all_y==0).sum()),
                                         "y=1": int((all_y==1).sum())},
            "pred_distribution_test":  {"p=0": int((all_p==0).sum()),
                                         "p=1": int((all_p==1).sum())},
            "prob_min":  round(float(all_pr.min()) if len(all_pr) else 0.0, 4),
            "prob_max":  round(float(all_pr.max()) if len(all_pr) else 0.0, 4),
            "prob_mean": round(float(all_pr.mean()) if len(all_pr) else 0.0, 4),
            "raw_y_pred_prob_first_50": [
                {"y": int(yy), "p": int(pp), "prob": round(float(prob), 4)}
                for yy, pp, prob in zip(all_y[:50], all_p[:50], all_pr[:50])
            ],
        },
    }


print("Training 12 MLP variants ...")
_t0 = time.perf_counter()
variant_results = {}
for letter, keys in VARIANTS.items():
    print(f"\n--- VARIANT {letter} ({len(keys)} scalars: {keys}) ---")
    variant_results[letter] = train_one_variant(letter, keys)
    m = variant_results[letter]["wikipedia_eval"]
    print(f"  AUROC {m['auc_roc']:.4f}  Acc {m['accuracy']:.4f}  F1 {m['f1']:.4f}  "
          f"input_dim {variant_results[letter]['input_dim']}")

DIAG["stage_timings_sec"]["all_variants_train"] = round(time.perf_counter() - _t0, 2)
DIAG["variants"] = variant_results
print(f"\n✓ all 12 variants trained in {DIAG['stage_timings_sec']['all_variants_train']} s")


In [ ]:
# =============================================================================
# BLOCK 5 (STAGE 5): CONSOLIDATED RESULTS DUMP
# =============================================================================
DIAG["timestamp_utc"] = (datetime.datetime.now(datetime.timezone.utc)
                          .replace(microsecond=0).isoformat().replace("+00:00", "Z"))
DIAG["host"] = ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ
                else ("colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
                      else "local"))
DIAG["stage_timings_sec"]["total"] = round(sum(DIAG["stage_timings_sec"].values()), 2)

with open(PATH_RESULTS, "w") as f:
    json.dump(DIAG, f, indent=2, default=str)
print(f"\n✓ wrote {PATH_RESULTS}  ({os.path.getsize(PATH_RESULTS)/1024:.1f} KB)")

# Final summary table
print("\n" + "=" * 96)
print(f"ALL-VARIANTS SUMMARY — {MODEL_TAG}")
print("=" * 96)
print(f"{'Variant':8s} {'Features':50s} {'AUROC':>8s} {'Acc':>7s} {'F1':>7s} {'Brier':>7s}")
print("-" * 96)
for letter, vd in DIAG["variants"].items():
    m = vd["wikipedia_eval"]
    feats = ", ".join(vd["feature_keys"]) if vd["feature_keys"] else "(canonical only)"
    feats = feats[:48] + ".." if len(feats) > 50 else feats
    print(f"{letter:8s} {feats:50s} {m['auc_roc']:8.4f} {m['accuracy']:7.4f} "
          f"{m['f1']:7.4f} {m['brier']:7.4f}")
print("=" * 96)
print("\nTimings:")
for k, v in DIAG["stage_timings_sec"].items():
    print(f"  {k:30s} {v:>8.2f} s")
print(f"\nFiles on disk:")
for p in [PATH_DATASET_FULL, PATH_DATASET_FEATURES, PATH_RESULTS]:
    if os.path.exists(p):
        print(f"  ✓ {p}   ({os.path.getsize(p)/1e6:.2f} MB)")
print(f"  ✓ {MODEL_TAG}_variant_<A..L>_best.pth   (12 files)")
print(f"\nPaste {PATH_RESULTS} back to the assistant for cross-variant audit.")
